<a href="https://colab.research.google.com/github/Janitha-Umeshan/Statistical-Learning-e23381/blob/main/Assignmet%2304.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Assignment: Building a Modular Data Sanitization & Exploration Engine

### Background
In real-world data science, 80% of the work is spent cleaning and exploring data. Most of this work is repetitive: checking for nulls, identifying outliers, and visualizing distributions. Your task is to build a **Reusable Python Class** named `DataInspector` and a supporting `PlottingMethods` class that can be imported into Google Colab to automate these tasks.

### The Objective
Develop an end-to-end tool for CSV data ingestion, advanced cleaning, feature engineering preparation, and high-level statistical visualization.

### Technical Requirements

#### 1. Data Ingestion & Sanitization
* **Colab Integration**: Implement `upload_data()` to handle local file uploads.
* **Garbage String Handling**: Automatically recognize and convert strings like `'?'`, `'n/a'`, `'NULL'`, and `' '` into actual `NaN` values.
* **Auto-Type Correction**: Force-convert columns to numeric types if the conversion does not result in an entirely null column.

#### 2. Structural Analysis & Cleaning
* **Data Summary**: Provide a method to display row/column counts, a preview of the first 20 rows, and a breakdown of numerical vs. categorical columns.
* **Intelligent Imputation**: Create a `handle_missing_values()` method supporting multiple strategies: `mean`, `median`, `mode`, or a `constant` value.
* **Duplicate & Outlier Management**:
    * Implement `remove_duplicates()` to prune exact row matches.
    * Develop an **IQR-based** outlier detection system (`handle_outliers`) that allows users to flag or automatically delete rows based on specific columns.
* **Targeted Deletion**: Implement interactive methods (`delete_rows`, `delete_columns`) that accept comma-separated user input to prune the dataset.

#### 3. Feature Engineering Preparation (Normalization)
* **Numeric Scaling**: Implement `extract_normalized_numeric_data()` supporting `minmax`, `standard` (Z-score), and `robust` (IQR-based) scaling.
* **Categorical Encoding**: Implement `extract_normalized_categorical_data()` supporting `onehot`, `ordinal`, and `uniform` (scaled 0-1) encoding.
* **Dataset Merging**: Provide a method to create a unified DataFrame containing original numeric data alongside encoded categorical data.

#### 4. Advanced Interactive Visualization (Plotly)
* **Univariate Subplots**: For numeric columns, generate a 3-panel subplot: **Horizontal Violin/Box**, **Scatter Plot** (Index vs Value), and **Histogram**.
* **Smart Relationships**: Build a `plot_relationship()` tool that detects types and chooses the correct chart:
    * **Num-Num**: Scatter with OLS Trendline.
    * **Cat-Num**: Box plot with all data points.
    * **Cat-Cat**: Grouped Bar chart.
* **Categorical Frequency**: Create bar charts displaying both raw counts and percentage labels.

#### 5. Deep Statistical Insights
* **Unified Heatmap**: Develop `plot_all_associations_heatmap()` to visualize relationships across *all* data types:
    * **Numeric-Numeric**: Pearson’s $r$.
    * **Categorical-Categorical**: Cramér’s $V$.
    * **Mixed (Num-Cat)**: Point-Biserial correlation or Eta (via ANOVA).

#### 6. Custom Modular Plotting
Implement a separate `PlottingMethods` class to handle granular chart generation (Bar, Pie, Histogram) that returns HTML-wrapped figures for flexible embedding.

### Submission Criteria
1.  **Object-Oriented Design**: All logic must be encapsulated within the `DataInspector` and `PlottingMethods` classes.
2.  **Clean Code**: Every method must include descriptive **Docstrings** and handle empty/None data gracefully.
3.  **Real-world Testing**: Demonstrate the tool using a dataset (e.g., Titanic) by performing a full flow: Upload $\rightarrow$ Impute $\rightarrow$ Normalize $\rightarrow$ Visualize Associations.

In [8]:

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from google.colab import files

from sklearn.preprocessing import (
    MinMaxScaler,
    StandardScaler,
    RobustScaler,
    OneHotEncoder,
    OrdinalEncoder
)

from scipy.stats import (
    chi2_contingency,
    pointbiserialr,
    f_oneway
)

import plotly.express as px
import plotly.graph_objects as go

from plotly.subplots import make_subplots

# Plotting Methods Class

class PlottingMethods:
    """
    Reusable Plotly chart generator.

    Returns HTML strings for easy embedding in dashboards,
    reports, web apps, notebooks, and Colab.
    """

    @staticmethod
    def fig_to_html(fig):
        return fig.to_html(full_html=False)

    @staticmethod
    def bar_chart(df, x, y=None, title="Bar Chart"):
        fig = px.bar(df, x=x, y=y, title=title)
        return PlottingMethods.fig_to_html(fig)

    @staticmethod
    def pie_chart(df, names, title="Pie Chart"):
        fig = px.pie(df, names=names, title=title)
        return PlottingMethods.fig_to_html(fig)

    @staticmethod
    def histogram(df, column, bins=30):
        fig = px.histogram(
            df,
            x=column,
            nbins=bins,
            title=f"Histogram - {column}"
        )
        return PlottingMethods.fig_to_html(fig)

# Data Inspector Class

class DataInspector:
    """
    Comprehensive data ingestion, cleaning,
    preprocessing and visualization utility.
    """

    GARBAGE_VALUES = [
        "?",
        "n/a",
        "na",
        "NULL",
        "null",
        " ",
        "",
        "None"
    ]

    def __init__(self, df=None):
        self.df = df

    # DATA INGESTION

    def upload_data(self):
        """
        Upload CSV file in Google Colab.
        Automatically sanitizes values and data types.
        """

        uploaded = files.upload()

        if len(uploaded) == 0:
            print("No file uploaded.")
            return None

        filename = list(uploaded.keys())[0]

        self.df = pd.read_csv(
            filename,
            na_values=self.GARBAGE_VALUES
        )

        self._auto_type_correction()

        print(f"Loaded: {filename}")
        print(f"Shape: {self.df.shape}")

        return self.df

    def load_dataframe(self, df):
        """
        Load existing dataframe directly.
        """
        self.df = df.copy()
        self._auto_type_correction()

    def _auto_type_correction(self):
        """
        Convert object columns to numeric whenever
        conversion does not produce all NaNs.
        """

        if self.df is None:
            return

        for col in self.df.columns:

            if self.df[col].dtype == object:

                converted = pd.to_numeric(
                    self.df[col],
                    errors="coerce"
                )

                if converted.notna().sum() > 0:
                    self.df[col] = converted

    # DATA SUMMARY

    def summary(self):
        """
        Display dataset overview.
        """

        if self.df is None:
            print("No dataset loaded.")
            return

        numeric = self.df.select_dtypes(
            include=np.number
        ).columns.tolist()

        categorical = self.df.select_dtypes(
            exclude=np.number
        ).columns.tolist()

        print("=" * 50)
        print("DATASET SUMMARY")
        print("=" * 50)

        print(f"Rows: {self.df.shape[0]}")
        print(f"Columns: {self.df.shape[1]}")

        print("\nNumeric Columns:")
        print(numeric)

        print("\nCategorical Columns:")
        print(categorical)

        print("\nFirst 20 Rows:")
        display(self.df.head(20))

    # MISSING VALUES

    def handle_missing_values(
        self,
        strategy="mean",
        constant_value=0
    ):
        """
        Strategies:
        mean
        median
        mode
        constant
        """

        if self.df is None:
            return

        for col in self.df.columns:

            if self.df[col].isna().sum() == 0:
                continue

            if strategy == "mean":

                if pd.api.types.is_numeric_dtype(
                    self.df[col]
                ):
                    self.df[col].fillna(
                        self.df[col].mean(),
                        inplace=True
                    )

            elif strategy == "median":

                if pd.api.types.is_numeric_dtype(
                    self.df[col]
                ):
                    self.df[col].fillna(
                        self.df[col].median(),
                        inplace=True
                    )

            elif strategy == "mode":

                self.df[col].fillna(
                    self.df[col].mode()[0],
                    inplace=True
                )

            elif strategy == "constant":

                self.df[col].fillna(
                    constant_value,
                    inplace=True
                )

        print("Missing values handled.")

    # DUPLICATES

    def remove_duplicates(self):

        if self.df is None:
            return

        before = len(self.df)

        self.df.drop_duplicates(inplace=True)

        after = len(self.df)

        print(
            f"Removed {before-after} duplicate rows."
        )

    # OUTLIERS

    def handle_outliers(
        self,
        columns,
        remove=False
    ):
        """
        IQR-based outlier detection.
        """

        if self.df is None:
            return

        outlier_rows = set()

        for col in columns:

            Q1 = self.df[col].quantile(.25)
            Q3 = self.df[col].quantile(.75)

            IQR = Q3 - Q1

            lower = Q1 - 1.5 * IQR
            upper = Q3 + 1.5 * IQR

            idx = self.df[
                (self.df[col] < lower)
                |
                (self.df[col] > upper)
            ].index

            outlier_rows.update(idx)

        print(
            f"Detected {len(outlier_rows)} outlier rows."
        )

        if remove:
            self.df.drop(
                list(outlier_rows),
                inplace=True
            )

            print("Outliers removed.")

        return list(outlier_rows)

    # DELETE ROWS/COLUMNS

    def delete_rows(self):

        rows = input(
            "Enter row indexes (comma separated): "
        )

        idx = [
            int(x.strip())
            for x in rows.split(",")
        ]

        self.df.drop(
            idx,
            inplace=True,
            errors="ignore"
        )

    def delete_columns(self):

        cols = input(
            "Enter columns (comma separated): "
        )

        cols = [
            c.strip()
            for c in cols.split(",")
        ]

        self.df.drop(
            columns=cols,
            inplace=True,
            errors="ignore"
        )

    # NORMALIZATION

    def extract_normalized_numeric_data(
        self,
        method="standard"
    ):

        numeric = self.df.select_dtypes(
            include=np.number
        )

        if numeric.empty:
            return pd.DataFrame()

        if method == "minmax":
            scaler = MinMaxScaler()

        elif method == "robust":
            scaler = RobustScaler()

        else:
            scaler = StandardScaler()

        transformed = scaler.fit_transform(
            numeric
        )

        return pd.DataFrame(
            transformed,
            columns=numeric.columns
        )

    def extract_normalized_categorical_data(
        self,
        method="onehot"
    ):

        cat = self.df.select_dtypes(
            exclude=np.number
        )

        if cat.empty:
            return pd.DataFrame()

        if method == "onehot":

            enc = OneHotEncoder(
                sparse_output=False,
                handle_unknown="ignore"
            )

            transformed = enc.fit_transform(cat)

            cols = enc.get_feature_names_out(
                cat.columns
            )

            return pd.DataFrame(
                transformed,
                columns=cols
            )

        elif method == "ordinal":

            enc = OrdinalEncoder()

            transformed = enc.fit_transform(cat)

            return pd.DataFrame(
                transformed,
                columns=cat.columns
            )

        elif method == "uniform":

            enc = OrdinalEncoder()

            transformed = enc.fit_transform(cat)

            scaler = MinMaxScaler()

            transformed = scaler.fit_transform(
                transformed
            )

            return pd.DataFrame(
                transformed,
                columns=cat.columns
            )

    def create_feature_engineering_dataset(
        self,
        numeric_method="standard",
        categorical_method="onehot"
    ):

        num = self.extract_normalized_numeric_data(
            numeric_method
        )

        cat = self.extract_normalized_categorical_data(
            categorical_method
        )

        return pd.concat(
            [num, cat],
            axis=1
        )

    # UNIVARIATE ANALYSIS

    def plot_numeric_distribution(
        self,
        column
    ):

        fig = make_subplots(
            rows=1,
            cols=3,
            subplot_titles=[
                "Violin / Box",
                "Scatter",
                "Histogram"
            ]
        )

        fig.add_trace(
            go.Violin(
                y=self.df[column],
                box_visible=True
            ),
            row=1,
            col=1
        )

        fig.add_trace(
            go.Scatter(
                x=self.df.index,
                y=self.df[column],
                mode="markers"
            ),
            row=1,
            col=2
        )

        fig.add_trace(
            go.Histogram(
                x=self.df[column]
            ),
            row=1,
            col=3
        )

        fig.update_layout(
            title=column,
            width=1400
        )

        fig.show()

    # SMART RELATIONSHIPS

    def plot_relationship(
        self,
        col1,
        col2
    ):

        is_num1 = pd.api.types.is_numeric_dtype(
            self.df[col1]
        )

        is_num2 = pd.api.types.is_numeric_dtype(
            self.df[col2]
        )

        if is_num1 and is_num2:

            fig = px.scatter(
                self.df,
                x=col1,
                y=col2,
                trendline="ols"
            )

        elif is_num1 != is_num2:

            cat = col1 if not is_num1 else col2
            num = col2 if not is_num2 else col1

            fig = px.box(
                self.df,
                x=cat,
                y=num,
                points="all"
            )

        else:

            cross = pd.crosstab(
                self.df[col1],
                self.df[col2]
            )

            fig = px.bar(
                cross,
                barmode="group"
            )

        fig.show()

    # CATEGORY FREQUENCIES

    def plot_categorical_frequency(
        self,
        column
    ):

        counts = self.df[column].value_counts()

        percentages = (
            counts
            /
            counts.sum()
            *
            100
        ).round(2)

        fig = go.Figure()

        fig.add_trace(
            go.Bar(
                x=counts.index.astype(str),
                y=counts.values,
                text=[
                    f"{p}%"
                    for p in percentages
                ]
            )
        )

        fig.update_layout(
            title=f"{column} Frequency"
        )

        fig.show()

    # ASSOCIATIONS

    @staticmethod
    def cramers_v(x, y):

        confusion = pd.crosstab(x, y)

        chi2 = chi2_contingency(
            confusion
        )[0]

        n = confusion.sum().sum()

        phi2 = chi2 / n

        r, k = confusion.shape

        return np.sqrt(
            phi2 /
            min(k - 1, r - 1)
        )

    @staticmethod
    def eta_squared(cat, num):

        groups = [
            num[cat == level]
            for level in np.unique(cat)
        ]

        f_stat, _ = f_oneway(*groups)

        n = len(num)

        return np.sqrt(
            f_stat /
            (f_stat + n)
        )

    def plot_all_associations_heatmap(self):

        cols = self.df.columns

        matrix = pd.DataFrame(
            index=cols,
            columns=cols,
            dtype=float
        )

        for c1 in cols:

            for c2 in cols:

                if c1 == c2:
                    matrix.loc[c1, c2] = 1
                    continue

                s1 = self.df[c1]
                s2 = self.df[c2]

                num1 = pd.api.types.is_numeric_dtype(
                    s1
                )

                num2 = pd.api.types.is_numeric_dtype(
                    s2
                )

                try:

                    if num1 and num2:

                        val = s1.corr(s2)

                    elif (not num1) and (not num2):

                        val = self.cramers_v(
                            s1,
                            s2
                        )

                    else:

                        if num1:
                            val = self.eta_squared(
                                s2,
                                s1
                            )
                        else:
                            val = self.eta_squared(
                                s1,
                                s2
                            )

                    matrix.loc[c1, c2] = val

                except:
                    matrix.loc[c1, c2] = np.nan

        fig = px.imshow(
            matrix,
            text_auto=True,
            color_continuous_scale="RdBu_r",
            title="Unified Association Heatmap"
        )

        fig.show()



In [10]:
import seaborn as sns

titanic = sns.load_dataset("titanic")

inspector = DataInspector(titanic)

# Summary
inspector.summary()

# Missing values
inspector.handle_missing_values(
    strategy="mode"
)

# Duplicates
inspector.remove_duplicates()

# Outliers
inspector.handle_outliers(
    columns=["age", "fare"],
    remove=False
)

# Standardized numeric data
numeric_scaled = (
    inspector.extract_normalized_numeric_data(
        method="standard"
    )
)

print(numeric_scaled.head())

# One-hot categorical encoding
cat_encoded = (
    inspector.extract_normalized_categorical_data(
        method="onehot"
    )
)

print(cat_encoded.head())

# Combined dataset
ml_dataset = (
    inspector.create_feature_engineering_dataset(
        numeric_method="standard",
        categorical_method="onehot"
    )
)

print(ml_dataset.shape)

# Distribution
inspector.plot_numeric_distribution(
    "fare"
)

# Relationships
inspector.plot_relationship(
    "fare",
    "age"
)

inspector.plot_relationship(
    "sex",
    "fare"
)

inspector.plot_relationship(
    "sex",
    "class"
)

# Frequency
inspector.plot_categorical_frequency(
    "embarked"
)

# Unified correlations
inspector.plot_all_associations_heatmap()

DATASET SUMMARY
Rows: 891
Columns: 15

Numeric Columns:
['survived', 'pclass', 'age', 'sibsp', 'parch', 'fare']

Categorical Columns:
['sex', 'embarked', 'class', 'who', 'adult_male', 'deck', 'embark_town', 'alive', 'alone']

First 20 Rows:


,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True
5,0,3,male,NaN,0,0,8.4583,Q,Third,man,True,NaN,Queenstown,no,True
6,0,1,male,54.0,0,0,51.8625,S,First,man,True,E,Southampton,no,True
7,0,3,male,2.0,3,1,21.0750,S,Third,child,False,NaN,Southampton,no,False
8,1,3,female,27.0,0,2,11.1333,S,Third,woman,False,NaN,Southampton,yes,False
9,1,2,female,14.0,1,0,30.0708,C,Second,child,False,NaN,Cherbourg,yes,False


Missing values handled.
Removed 113 duplicate rows.
Detected 119 outlier rows.
   survived    pclass       age     sibsp     parch      fare
0 -0.842550  0.889003 -0.509883  0.479881 -0.499547 -0.528481
1  1.186874 -1.451587  0.644663  0.479881 -0.499547  0.696162
2  1.186874  0.889003 -0.221246 -0.531901 -0.499547 -0.515571
3  1.186874 -1.451587  0.428185  0.479881 -0.499547  0.348404
4 -0.842550  0.889003  0.428185 -0.531901 -0.499547 -0.513180
   sex_female  sex_male  embarked_C  embarked_Q  embarked_S  class_First  \
0         0.0       1.0         0.0         0.0         1.0          0.0   
1         1.0       0.0         1.0         0.0         0.0          1.0   
2         1.0       0.0         0.0         0.0         1.0          0.0   
3         1.0       0.0         0.0         0.0         1.0          1.0   
4         0.0       1.0         0.0         0.0         1.0          0.0   

   class_Second  class_Third  who_child  who_man  ...  deck_E  deck_F  deck_G  \
0          